In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline 

In [2]:
train_df = pd.read_csv("C:/Users/Ainsley/Projects/datasets/fair-face-img-margin-125trainval/fairface_label_train-mini.csv")
test_df = pd.read_csv("C:/Users/Ainsley/Projects/datasets/fair-face-img-margin-125trainval/fairface_label_val-mini.csv")

In [3]:
100*train_df.groupby(['race']).count()[['file']]/train_df.groupby(['race']).count()[['file']].sum()

,file
race,
Black,14.743236
East Asian,13.597460
Indian,14.563777
Latino_Hispanic,15.571507
Middle Eastern,10.574268
Southeast Asian,11.844285
White,19.105467


In [4]:
train_df.head()

,file,age,gender,race,service_test
0,train/1.jpg,50-59,Male,East Asian,True
1,train/2.jpg,30-39,Female,Indian,False
2,train/3.jpg,3-Sep,Female,Black,False
3,train/4.jpg,20-29,Female,Indian,True
4,train/5.jpg,20-29,Female,Indian,True


In [5]:
train_df.count

<bound method DataFrame.count of                 file     age  gender             race  service_test
0        train/1.jpg   50-59    Male       East Asian          True
1        train/2.jpg   30-39  Female           Indian         False
2        train/3.jpg   3-Sep  Female            Black         False
3        train/4.jpg   20-29  Female           Indian          True
4        train/5.jpg   20-29  Female           Indian          True
...              ...     ...     ...              ...           ...
7239  train/7240.jpg   3-Sep    Male           Indian         False
7240  train/7241.jpg   30-39    Male            White         False
7241  train/7242.jpg   40-49  Female  Southeast Asian          True
7242  train/7243.jpg  Oct-19    Male   Middle Eastern          True
7243  train/7244.jpg  Oct-19  Female   Middle Eastern          True

[7244 rows x 5 columns]>

In [6]:
from keras.preprocessing import image
target_size = (224, 224)
def getImagePixels(file):
    prepend = "C:/Users/Ainsley/Projects/datasets/fair-face-img-margin-125trainval/"
    img = image.load_img(prepend + file, grayscale=False, target_size=target_size)
    x = image.img_to_array(img).reshape(1, -1)[0]
    #print(prepend+file)
    return x

Using TensorFlow backend.


In [7]:
from tqdm.notebook import tqdm
tqdm.pandas()
import ipywidgets as widgets 

D:\Downloads\Ana\envs\tensorflow\lib\site-packages\tqdm\std.py:668: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [8]:
train_df['pixels'] = train_df['file'].progress_apply(getImagePixels)

In [9]:
test_df['pixels'] = test_df['file'].progress_apply(getImagePixels)

In [10]:
train_df.head()

,file,age,gender,race,service_test,pixels
0,train/1.jpg,50-59,Male,East Asian,True,"[8.0, 8.0, 10.0, 9.0, 9.0, 11.0, 10.0, 8.0, 11..."
1,train/2.jpg,30-39,Female,Indian,False,"[129.0, 127.0, 104.0, 127.0, 125.0, 102.0, 123..."
2,train/3.jpg,3-Sep,Female,Black,False,"[216.0, 171.0, 174.0, 212.0, 167.0, 170.0, 206..."
3,train/4.jpg,20-29,Female,Indian,True,"[42.0, 47.0, 50.0, 42.0, 47.0, 50.0, 41.0, 46...."
4,train/5.jpg,20-29,Female,Indian,True,"[44.0, 39.0, 35.0, 44.0, 39.0, 35.0, 43.0, 40...."


## input features
Pixels are stored as a list. We need to reshape each line to (224, 224, 3). Besides, inputs should be normalized in neural networks because of activation functions. This is going to be input feature we will pass to the network as input.



In [11]:
import time
tic = time.time()
train_features = []

for i in range(0, train_df.shape[0]):
    train_features.append(train_df['pixels'].values[i])
    

train_features = np.array(train_features)
train_features = train_features.reshape(train_features.shape[0], 224, 224, 3)

train_features /= 255

toc = time.time()

print("train_features converted to numpy in ",toc-tic,"seconds")

train_features converted to numpy in  450.36223435401917 seconds


In [12]:
tic = time.time()

test_features = []

for i in range(0, test_df.shape[0]):
    test_features.append(test_df['pixels'].values[i])

test_features = np.array(test_features)
test_features = test_features.reshape(test_features.shape[0], 224, 224, 3)
 
test_features = test_features / 255

toc = time.time()

print("test_features converted to numpy in ",toc-tic,"seconds")

test_features converted to numpy in  89.57714557647705 seconds


In [13]:
print(train_features.shape)

(7244, 224, 224, 3)


In [14]:
train_label = train_df[['race']] 
test_label = test_df[['race']]

races = train_df['race'].unique()
 
for j in range(len(races)): #label encoding
    current_race = races[j]
    print("replacing ",current_race," to ", j+1)
    train_label['race'] = train_label['race'].replace(current_race, str(j+1))
    test_label['race'] = test_label['race'].replace(current_race, str(j+1))
 
train_label = train_label.astype({'race': 'int32'})
test_label = test_label.astype({'race': 'int32'})
 
train_target = pd.get_dummies(train_label['race'], prefix='race')
test_target = pd.get_dummies(test_label['race'], prefix='race')

replacing  East Asian  to  1


D:\Downloads\Ana\envs\tensorflow\lib\site-packages\ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':
D:\Downloads\Ana\envs\tensorflow\lib\site-packages\ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.


replacing  Indian  to  2
replacing  Black  to  3
replacing  White  to  4
replacing  Middle Eastern  to  5
replacing  Latino_Hispanic  to  6
replacing  Southeast Asian  to  7


## Train and validation split


In [15]:
tic = time.time()
train_x, val_x, train_y, val_y = train_test_split(
    train_features, train_target.values
    , test_size=0.12, random_state=17
)
toc = time.time()
print("train-test split in ",toc-tic,"seconds")

train-test split in  567.1412975788116 seconds


## Base Model
We use VGG-Face for transfer learning

In [16]:
from keras.models import Model, Sequential
from keras.layers import Input, Convolution2D, ZeroPadding2D, MaxPooling2D, Flatten, Dense, Dropout, Activation
from keras.optimizers import Adam

model = Sequential()
model.add(ZeroPadding2D((1,1),input_shape=(224,224, 3)))
model.add(Convolution2D(64, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
 
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(128, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
 
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
 
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
 
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
 
model.add(Convolution2D(4096, (7, 7), activation='relu'))
model.add(Dropout(0.5))
model.add(Convolution2D(4096, (1, 1), activation='relu'))
model.add(Dropout(0.5))
model.add(Convolution2D(2622, (1, 1)))
model.add(Flatten())
model.add(Activation('softmax'))
 
#related blog post: https://sefiks.com/2018/08/06/deep-face-recognition-with-keras/
model.load_weights('vgg_face_weights.h5')

## Transfer Learning
Its early layers can detect some facial patterns already. We do not have to train it from scratch. Because we do not have millions of train set instances. We can lock its early layers and expect the late layers to learn.

In [17]:
#freeze all layers of VGG-Face except last 7 one
for layer in model.layers[:-7]:
    layer.trainable = False

In this way, its all layers except the last 7 one are locked and its weights will not be updated. We expect its last 7 layers to learn something.

The original VGG-Face network has 2622 outputs but here we need just 6 outputs related to races. We will customize the VGG-Face here and it is going to be VGG-Race now.

In [18]:
num_of_classes = len(races)
base_model_output = Sequential()
base_model_output = Convolution2D(num_of_classes, (1, 1), name='predictions')(model.layers[-4].output)
base_model_output = Flatten()(base_model_output)
base_model_output = Activation('softmax')(base_model_output)
 
race_model = Model(inputs=model.input, outputs=base_model_output)

## Training
Instead of feeding all train data, I prefer to feed it as batches. I got the best result for 16.384 (2^14) batch size. I feed randomly selected 16K instances in every epoch. If validation loss would not decrease for 50 rounds, then training should be terminated to avoid overfitting.

In [ ]:
from keras.callbacks import ModelCheckpoint

epochs = 500
race_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(), metrics=['accuracy'])
 
checkpointer = ModelCheckpoint(filepath='race_model_single_batch.hdf5', 
                               monitor = "val_loss", verbose=1, save_best_only=True, mode = 'auto')
 
batch_size = pow(2, 14); patience = 50
last_improvement = 0; best_iteration = 0
loss = 1000000 #initialize as a large value
val_scores=[]
train_scores=[]

for i in range(0, epochs):
    print("Epoch ", i, ". ", end='')
    ix_train = np.random.choice(train_x.shape[0], size=batch_size)
 
    score = race_model.fit(
        train_x[ix_train], train_y[ix_train]
        , epochs=1
        , validation_data=(val_x, val_y)
        , callbacks=[checkpointer]
    )
 
    val_loss = score.history['val_loss'][0]; train_loss = score.history['loss'][0]
    val_scores.append(val_loss); train_scores.append(train_loss)
 
    if val_loss < loss:
        loss = val_loss * 1
        last_improvement = 0
        best_iteration = i * 1
    else:
        last_improvement = last_improvement + 1
        print("try to decrease val loss for ",patience - last_improvement," epochs more")
 
    if last_improvement == patience:
        print("there is no loss decrease in validation for ",patience," epochs. early stopped")
        break

Epoch  0 . 

## Loss
train loss decreased while validation loss increased when epoch > 7 (in this case, indicating overfitting)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(val_scores[0:best_iteration+1], label='val_loss')
plt.plot(train_scores[0:best_iteration+1], label='train_loss')
plt.legend(loc='upper right')
plt.show()

## Evaluation
We train the network with train data set and use validation set to apply early stop. Epoch is the best iteration for validation set actually. However, network could memorize the validation set and it could still be overfitted. That’s why, we haven’t feed test set to the network yet. We expect that test and validation loss should be close if the model is robust.

In [ ]:
# test_label = test_df[['race']]
# races = train_df['race'].unique()
# for j in range(len(races)): #label encoding
#     current_race = races[j]
#     print("replacing ",current_race," to ", j+1)
#     test_label['race'] = test_label['race'].replace(current_race, str(j+1))
 
# test_label = test_label.astype({'race': 'int32'})
 
# test_target = pd.get_dummies(test_label['race'], prefix='race')


In [ ]:
from keras.models import load_model
race_model = load_model("race_model_single_batch.hdf5")
race_model.save_weights('race_model_single_batch.h5')

test_perf = race_model.evaluate(test_features, test_target.values, verbose=1)
print(test_perf)
 
validation_perf = race_model.evaluate(val_x, val_y, verbose=1)
print(validation_perf)
 
abs(validation_perf[0] - test_perf[0])

## Prediction


In [ ]:
prediction_classes=[]
actual_classes=[]
predictions = race_model.predict(test_features)
for i in range(0, predictions.shape[0]):
    prediction = np.argmax(predictions[i])
    prediction_classes.append(races[prediction])
 
    actual = np.argmax(test_target.values[i])
    actual_classes.append(races[actual])
 
    if i in [
        375, 470, 750, 758, 875, 992, 1061, 2181, 2255, 4725, 4944 #latino
        , 124, 339, 762, 913, 1340, 1363, 2205 #black
        , 33, 83, 237, 609, 817, 1223, 1377 #asian
        , 109, 203, 899, 1094, 1180, 1250, 1395, 1556 #indian
        , 638, 718, 1088, 1460, 4396, 4477 #middle eastern
        , 413, 447, 573, 649, 723, 1258, 1274, 1430, 1485 #white
        , 17, 235, #misclassified
    ]:
        print(i)
        print("Actual: ",races[actual])
        print("Predicted: ",races[prediction])
        img = (test_df.iloc[i]['pixels'].reshape([224, 224, 3])) / 255
        plt.imshow(img); plt.show()

## Confusion matrix
Accuracy does not mean anything for classification problems. We need precision and recall values. Confusion matrix is the best way to monitor the success of your model.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sn
 
cm = confusion_matrix(actual_classes, prediction_classes)
df_cm = pd.DataFrame(cm, index=races, columns=races)
sn.heatmap(df_cm, annot=True,annot_kws={"size": 10})


## Predicting custom images
We can predict the ethnicity for custom images as well.


In [ ]:
races = ['East Asian', 'Indian', 'Black', 'White', 'Middle Eastern', 'Latino_Hispanic', 'Southeast Asian']

In [ ]:
demo_set = ['richard.png','sundar-pichai.jpg','Jian_Yang.jpg']
 
for file in demo_set:
    path = 'demo/%s' % (file)
    img = image.load_img(path, grayscale=False, target_size=target_size)
    img = image.img_to_array(img).reshape(1, -1)[0]
    img = img.reshape(224, 224, 3)
    img = img / 255
 
    plt.imshow(img)
    plt.show()
 
    img = np.expand_dims(img, axis=0) 
 
    prediction_proba = race_model.predict(img)
 
    print("Prediction: ",races[np.argmax(prediction_proba)])
    print("---------------------------")